# 01.7 训练循环（Training Loop）

这一节把前面几节真正串起来。  

你前面已经学过：  

- 张量（tensors）
- 自动微分（autograd）
- 数据集和数据加载器（datasets and data loaders）
- 模型定义（model definition）
- 损失函数和优化器（loss functions and optimizers）

training loop 就是把这些模块按正确顺序组织起来。  


## 学习目标

学完后你应该能

1. 解释完整训练循环的步骤
2. 正确使用 `model.train()` 和 `model.eval()`
3. 正确组织 `zero_grad()`、`backward()`、`step()`
4. 区分训练阶段和验证阶段
5. 记录 loss 和 accuracy
6. 写出一个可复用的最小训练框架

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 1. 准备一个最小分类数据集

为了专注于训练循环本身，这里直接构造一个二维二分类数据集。  


In [ ]:
torch.manual_seed(0)

class0 = torch.randn(80, 2) * 0.6 + torch.tensor([-1.2, -1.0])
class1 = torch.randn(80, 2) * 0.6 + torch.tensor([1.2, 1.0])

X = torch.cat([class0, class1], dim=0).float()
y = torch.cat([
    torch.zeros(len(class0), dtype=torch.long),
    torch.ones(len(class1), dtype=torch.long),
])

perm = torch.randperm(len(X))
X = X[perm]
y = y[perm]

split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)
print("X_val.shape =", X_val.shape)
print("y_val.shape =", y_val.shape)

In [ ]:
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("前 3 个标签 / first 3 labels:", yb[:3])

## 2. 定义模型

这里用一个非常简单的 MLP。  


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, in_features=2, hidden_features=8, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = SmallClassifier()
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print(loss_fn)
print(optimizer)

## 3. 训练循环的最小骨架

Training phase 的固定顺序通常是：  

1. `model.train()`
2. 遍历 batch
3. `optimizer.zero_grad()`
4. `pred = model(xb)`
5. `loss = loss_fn(pred, yb)`
6. `loss.backward()`
7. `optimizer.step()`

Validation phase 通常是：  

1. `model.eval()`
2. `with torch.no_grad():`
3. 只前向，不更新参数

In [ ]:
def compute_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    acc = (preds == targets).float().mean().item()
    return acc


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += compute_accuracy(logits, yb)
        num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += loss.item()
            total_acc += compute_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 4. 训练几个 epoch

一个 `epoch` 表示完整看一遍训练集。  


In [ ]:
history = []

for epoch in range(1, 16):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
print("最后一条记录 / last history record:")
print(history[-1])

## 5. `train()` 和 `eval()` 为什么重要

当前这个小模型没有 `Dropout` 或 `BatchNorm`，所以两者差别还不明显。  

但从工程习惯上，这两个模式必须养成。  

- `model.train()`：训练模式（training mode）
- `model.eval()`：评估模式（evaluation mode）

## 6. 小练习

这几题的重点不是写很多代码，而是把训练循环的结构真正记住。  


In [ ]:
# 练习 1
# 用一句话说明，为什么训练阶段通常要先调用 optimizer.zero_grad()。
# In one sentence, explain why training usually starts with optimizer.zero_grad().

参考回答

因为 `PyTorch` 默认会累积梯度，所以每个 batch 开始前通常要先清掉上一次的梯度。  


In [ ]:
# 练习 2
# 补全下面的函数
#
# 目标
# 给定 logits 和 targets，返回 batch accuracy。
# Given logits and targets, return the batch accuracy.

def batch_accuracy(logits, targets):
    # TODO
    pass


# logits = torch.tensor([[2.0, 1.0], [0.2, 0.8], [1.5, 0.1]])
# targets = torch.tensor([0, 1, 0])
# print(batch_accuracy(logits, targets))

In [ ]:
# 练习 2 参考答案

def batch_accuracy_solution(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


logits = torch.tensor([[2.0, 1.0], [0.2, 0.8], [1.5, 0.1]])
targets = torch.tensor([0, 1, 0])
print(batch_accuracy_solution(logits, targets))

In [ ]:
# 练习 3
# 解释下面两行代码分别做什么。
# Explain what each of the two lines below does.
#
# model.train()
# model.eval()
#
# 请用你自己的话写 2 句话。
# Write 2 sentences in your own words.

## 7. 小结

训练循环的核心顺序一定要形成肌肉记忆：  

1. 取一个 batch
2. 前向传播
3. 计算损失
4. 清零梯度
5. 反向传播
6. 更新参数

你现在应该能回答

1. 为什么训练和验证阶段的代码不一样？
2. 为什么 `zero_grad()`、`backward()`、`step()` 的顺序不能乱？
3. 一个 epoch 的含义是什么？

下一步建议

- 进入模型保存与推理 notebook，把训练好的模型真正保存下来并重新使用